<a href="https://colab.research.google.com/github/asmmorshed101/BanglaTextGeneration/blob/main/BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch sentence-transformers PyMuPDF faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.5 MB/s eta 0:00:00


In [2]:
pip install transformers torch sentence-transformers PyMuPDF

In [3]:
pip install faiss-cpu

In [4]:
import fitz  # PyMuPDF
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import numpy as np

In [5]:
# Step 1: Read PDF
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""

    for page in doc:
        full_text += page.get_text()

    return full_text

In [6]:
# Step 2: Split Text into Chunks
def split_text(text, chunk_size=500):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])

    return chunks

In [7]:
# Step 3: Load Embedding Model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# Step 4: Create Embeddings
def create_embeddings(chunks):
    embeddings = embedding_model.encode(chunks)
    return embeddings

In [9]:
# Step 5: Find Most Relevant Chunk

def find_most_relevant_chunk(question, chunks, chunk_embeddings):
    question_embedding = embedding_model.encode([question])[0]

    similarities = []

    for emb in chunk_embeddings:
        similarity = np.dot(question_embedding, emb) / (
            np.linalg.norm(question_embedding) * np.linalg.norm(emb)
        )
        similarities.append(similarity)

    best_index = np.argmax(similarities)
    return chunks[best_index]

In [10]:
# Step 6: Load BERT QA Model

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [11]:
# Step 7: Ask Question
def ask_question(question, pdf_path):
    text = extract_text_from_pdf(pdf_path)
    chunks = split_text(text)
    chunk_embeddings = create_embeddings(chunks)

    relevant_chunk = find_most_relevant_chunk(
        question,
        chunks,
        chunk_embeddings
    )

    result = qa_pipeline(
        question=question,
        context=relevant_chunk
    )

    return {
        "question": question,
        "answer": result['answer'],
        "score": result['score'],
        "context": relevant_chunk
    }

In [14]:
# Example Usage

pdf_file = "samplefile.pdf"

while True:
    question = input("Ask a question: ")

    if question.lower() == "exit":
        break

    response = ask_question(question, pdf_file)

    print("\nAnswer:", response['answer'])
    print("Confidence Score:", response['score'])
    print("\nRelevant Context:")
    print(response['context'])
    print("\n" + "-"*50)

Ask a question: What is the capital city of Bangladesh?

Answer: Dhaka
Confidence Score: 0.9691190529847518

Relevant Context:
Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, 
and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the country’s development. 
 
One of the most important natural features of Bangladesh is the Padma River, which is often 
referred to as the lifeline

--------------------------------------------------
Ask a question: Which river is considered the lifeline of Bangladesh?

Answer: Padma River
Confidence Score: 0.8209657958650496

Relevant Context:
Bangladesh is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, 
and vast river systems. The capital city of Bangladesh is Dhaka. Dhaka 

KeyboardInterrupt: Interrupted by user